# 08 - Aplikasi Web Interaktif: CITRA SERUPA
### Antarmuka Pengguna (UI) Rekomendasi Pakaian Berbasis Kemiripan Visual (Gradio Web App)

---

**Tujuan Notebook:**
Notebook ini mengubah seluruh model deep learning dan basis data fitur yang telah kita bangun menjadi **Aplikasi Web Interaktif yang siap dipakai oleh pengguna umum / dosen penguji**:
1. Pengguna cukup mengunggah foto pakaian (lewat kamera HP atau laptop).
2. Sistem secara instan mengekstrak fitur visualnya menggunakan **ResNet-50**.
3. Sistem menghitung **Cosine Similarity** terhadap seluruh katalog di Google Drive.
4. Menampilkan hasil **Top-K Rekomendasi** dalam galeri foto interaktif yang indah lengkap dengan skor kemiripan (%).
5. Menghasilkan **Tautan Publik Gratis (Public Link `gradio.live`)** sehingga aplikasi bisa dibuka dari perangkat mana pun (termasuk HP dosen/teman) tanpa perlu instalasi!

## 1. Hubungkan Google Drive & Instalasi Gradio

In [ ]:
# 1. Instalasi Gradio untuk antarmuka web interaktif
!pip install -q gradio

from google.colab import drive
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import gradio as gr

# 2. Hubungkan Google Drive
drive.mount('/content/drive')

# Deteksi direktori proyek
base_dir = '/content/drive/MyDrive/CITRA-SERUPA'
if not os.path.exists(base_dir):
    base_dir = '/content/drive/MyDrive/CITRA_SERUPA'

dataset_dir = None
images_dir = None
for d in ['dataset', 'dataset ']:
    cand = os.path.join(base_dir, d, 'images')
    if os.path.exists(cand) and len(os.listdir(cand)) > 0:
        dataset_dir = os.path.join(base_dir, d)
        images_dir = cand
        break

features_dir = os.path.join(base_dir, 'features')
csv_path = os.path.join(dataset_dir, 'metadata.csv')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 65)
print("⏳ Memuat basis data fitur dan model ekstraktor...")
embeddings = np.load(os.path.join(features_dir, 'embeddings.npy'))
image_names = np.load(os.path.join(features_dir, 'image_names.npy'))
item_ids = np.load(os.path.join(features_dir, 'item_ids.npy'))
df_meta = pd.read_csv(csv_path)

# 3. Siapkan Model Ekstraktor ResNet-50
feature_extractor = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
feature_extractor.fc = nn.Identity()
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# Pipeline Preprocessing Kueri
transform_query = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"✅ Sistem Siap: {len(image_names):,} gambar di galeri | Perangkat: {device}")
print("=" * 65)

## 2. Logika Mesin Pencarian Gambar (Backend Function)

In [ ]:
def search_similar_fashion(input_image, top_k=10):
    """
    Fungsi utama yang menerima foto kueri dari pengguna dan mengembalikan galeri foto serupa.
    """
    if input_image is None:
        return [], "⚠️ Silakan unggah atau pilih foto pakaian terlebih dahulu!"
    
    # Konversi ke format PIL RGB
    if not isinstance(input_image, Image.Image):
        input_image = Image.fromarray(input_image)
    if input_image.mode != 'RGB':
        input_image = input_image.convert('RGB')
    
    # 1. Preprocessing dan Forward Pass melalui ResNet-50
    tensor = transform_query(input_image).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = feature_extractor(tensor)
        feat = F.normalize(feat, p=2, dim=1)
        query_vec = feat.cpu().numpy()[0]
    
    # 2. Hitung Cosine Similarity terhadap seluruh 3.608 gambar katalog
    scores = np.dot(embeddings, query_vec)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    # 3. Format hasil untuk komponen galeri Gradio
    gallery_results = []
    details_text = f"### 🎯 Hasil Pencarian Rekomendasi (Top-{top_k}):\n\n"
    
    for rank, idx in enumerate(top_indices, 1):
        img_name = image_names[idx]
        item_id = item_ids[idx]
        score_pct = scores[idx] * 100
        img_path = os.path.join(images_dir, img_name)
        
        caption = f"Rank #{rank} ({score_pct:.1f}%)\n{item_id}"
        if os.path.exists(img_path):
            rec_img = Image.open(img_path)
            gallery_results.append((rec_img, caption))
        
        details_text += f"* **Rank #{rank:02d}:** `{img_name}` | **Kemiripan:** `{score_pct:.2f}%` | **Produk:** `{item_id}`\n"
    
    return gallery_results, details_text

print("✅ Logika pencarian visual siap dihubungkan ke UI!")

## 3. Peluncuran Aplikasi Web Interaktif (Gradio Web UI)

In [ ]:
# Buat antarmuka web modern dengan Gradio Blocks
theme = gr.themes.Soft(primary_hue="emerald")

with gr.Blocks(theme=theme, title="CITRA SERUPA — Rekomendasi Busana") as app:
    gr.Markdown(
        """
        # 👗 CITRA SERUPA: Sistem Rekomendasi Busana Berdasarkan Kemiripan Visual
        ### *Content-Based Image Retrieval (CBIR) Menggunakan Deep Learning ResNet-50*
        Unggah foto pakaian (baju, celana, dress) dari perangkat Anda, dan sistem AI akan merekomendasikan busana paling mirip dari koleksi galeri secara instan!
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="pil", label="📤 Unggah Foto Pakaian (Input Kueri)")
            slider_k = gr.Slider(minimum=5, maximum=20, value=10, step=1, label="Jumlah Rekomendasi (Top-K)")
            btn_search = gr.Button("🔍 Cari Rekomendasi Serupa", variant="primary", size="lg")
            gr.Markdown("💡 *Tips: Anda bisa mengunggah foto apa saja dari internet, kamera HP, atau katalog.*")
            
        with gr.Column(scale=2):
            output_gallery = gr.Gallery(
                label="🖼️ Galeri Pakaian Paling Serupa (Hasil Rekomendasi)",
                columns=5,
                height="auto",
                object_fit="contain"
            )
            output_details = gr.Markdown(label="Rincian Skor")
            
    # Hubungkan tombol pencarian ke backend
    btn_search.click(
        fn=search_similar_fashion,
        inputs=[input_img, slider_k],
        outputs=[output_gallery, output_details]
    )

# Luncurkan aplikasi dengan tautan publik gratis (share=True)
print("🚀 Meluncurkan Aplikasi Web CITRA SERUPA...")
app.launch(share=True, debug=True)